In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_1.py ──
"""
Shared infrastructure for Exercise 1 — The Complete Autoencoder Family.

Contains: data loading, visualisation helpers, training loop, engine setup.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker
from kailash_ml import ModelRegistry


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

# Output directory for all visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex1_autoencoders"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Fashion-MNIST (full 60K)
# ════════════════════════════════════════════════════════════════════════

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "fashion_mnist"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DIM = 28 * 28
LATENT_DIM = 16
EPOCHS = 10


def load_fashion_mnist() -> (
    tuple[
        torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, DataLoader, DataLoader
    ]
):
    """Load Fashion-MNIST and return flat/image tensors + loaders.

    Returns:
        (X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader)
    """
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_img = torch.stack([train_set[i][0] for i in range(len(train_set))])
    X_img = X_img.to(device).float()
    X_flat = X_img.reshape(len(X_img), -1)

    X_test_img = torch.stack([test_set[i][0] for i in range(len(test_set))])
    X_test_img = X_test_img.to(device).float()
    X_test_flat = X_test_img.reshape(len(X_test_img), -1)

    flat_loader = DataLoader(TensorDataset(X_flat), batch_size=256, shuffle=True)
    img_loader = DataLoader(TensorDataset(X_img), batch_size=256, shuffle=True)

    print(
        f"Fashion-MNIST loaded: {len(X_img)} train + {len(X_test_img)} test images, "
        f"shape {tuple(X_img.shape[1:])}, pixel range [{X_img.min():.2f}, {X_img.max():.2f}]"
    )

    return X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader


def get_fashion_mnist_labels() -> tuple[torch.Tensor, torch.Tensor]:
    """Return train and test label tensors."""
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=True, download=True
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=False, download=True
    )
    train_labels = torch.tensor([train_set[i][1] for i in range(len(train_set))])
    test_labels = torch.tensor([test_set[i][1] for i in range(len(test_set))])
    return train_labels, test_labels


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved for callers."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_autoencoders.db"
    registry_db = "sqlite:///mlfp05_autoencoders_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_autoencoders", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION UTILITIES — "Seeing Is Believing"
# ════════════════════════════════════════════════════════════════════════


def show_reconstruction(model, test_data, title, n=10, is_conv=False):
    """Show original vs reconstructed images side by side."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n].to(device)
        result = model(x)
        x_hat = result[0]

    fig, axes = plt.subplots(2, n, figsize=(15, 3))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n):
        if is_conv:
            orig = x[i].cpu().squeeze()
            recon = x_hat[i].cpu().squeeze()
        else:
            orig = x[i].cpu().reshape(28, 28)
            recon = x_hat[i].cpu().reshape(28, 28)

        axes[0, i].imshow(orig, cmap="gray")
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("Original", fontsize=9)

        axes[1, i].imshow(recon.clamp(0, 1), cmap="gray")
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_title("Reconstructed", fontsize=9)

    plt.tight_layout()
    fname = (
        OUTPUT_DIR
        / f"ex1_{title.lower().replace(' ', '_').replace('(', '').replace(')', '')}.png"
    )
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_denoising_grid(model, clean_data, title, n=10, sigma=0.3):
    """3-row grid: original, noisy input, cleaned output."""
    model.eval()
    with torch.no_grad():
        clean = clean_data[:n].to(device)
        noisy = torch.clamp(clean + sigma * torch.randn_like(clean), 0.0, 1.0)
        result = model(noisy)
        cleaned = result[0]

    fig, axes = plt.subplots(3, n, figsize=(15, 4.5))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    row_labels = ["Original", "Noisy Input", "Cleaned Output"]

    for i in range(n):
        for row, data in enumerate([clean, noisy, cleaned]):
            img = data[i].cpu().reshape(28, 28)
            axes[row, i].imshow(img.clamp(0, 1), cmap="gray")
            axes[row, i].axis("off")
            if i == 0:
                axes[row, i].set_title(row_labels[row], fontsize=9)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_denoising_ae.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_activation_sparsity(model, test_data, title="Sparse AE Activations"):
    """Histogram of hidden-layer activations showing sparsity."""
    model.eval()
    with torch.no_grad():
        x = test_data[:1000].to(device)
        h = model.encoder(x)

    activations = h.cpu().numpy().flatten()

    _, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(activations, bins=100, color="steelblue", edgecolor="white", alpha=0.8)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Activation Value")
    ax.set_ylabel("Frequency")
    pct_near_zero = (np.abs(activations) < 0.1).mean() * 100
    ax.annotate(
        f"{pct_near_zero:.1f}% of activations near zero",
        xy=(0.95, 0.95),
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"),
    )
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_sparse_activations.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_interpolation(model, test_data, title, n_steps=10, is_conv=False):
    """Morph between two images via latent space interpolation."""
    model.eval()
    with torch.no_grad():
        x1 = test_data[0:1].to(device)
        x2 = test_data[5:6].to(device)
        z1 = model.encoder(x1)
        z2 = model.encoder(x2)

        alphas = torch.linspace(0, 1, n_steps).to(device)
        interpolated = []
        for alpha in alphas:
            z = (1 - alpha) * z1 + alpha * z2
            x_hat = model.decoder(z)
            interpolated.append(x_hat)

    fig, axes = plt.subplots(1, n_steps + 2, figsize=(16, 2))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    src_img = x1[0].cpu().reshape(28, 28) if not is_conv else x1[0].cpu().squeeze()
    axes[0].imshow(src_img, cmap="gray")
    axes[0].set_title("Start", fontsize=8)
    axes[0].axis("off")

    for i, x_hat in enumerate(interpolated):
        img = x_hat[0].cpu()
        img = img.reshape(28, 28) if not is_conv else img.squeeze()
        axes[i + 1].imshow(img.clamp(0, 1), cmap="gray")
        axes[i + 1].set_title(f"{alphas[i]:.1f}", fontsize=7)
        axes[i + 1].axis("off")

    tgt_img = x2[0].cpu().reshape(28, 28) if not is_conv else x2[0].cpu().squeeze()
    axes[-1].imshow(tgt_img, cmap="gray")
    axes[-1].set_title("End", fontsize=8)
    axes[-1].axis("off")

    plt.tight_layout()
    fname = OUTPUT_DIR / f"ex1_{title.lower().replace(' ', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_generated_samples(model, title="VAE Generated Samples", grid_size=8):
    """Grid of images sampled from the VAE's learned prior N(0, I)."""
    model.eval()
    n = grid_size * grid_size
    with torch.no_grad():
        samples = model.sample(n).cpu()

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    for i in range(grid_size):
        for j in range(grid_size):
            idx = i * grid_size + j
            axes[i, j].imshow(samples[idx].reshape(28, 28).clamp(0, 1), cmap="gray")
            axes[i, j].axis("off")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_generated_samples.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_traversal(
    model, test_data, title="VAE Latent Traversal", n_dims=5, n_steps=11
):
    """Vary one latent dimension at a time and observe what changes."""
    model.eval()
    with torch.no_grad():
        x = test_data[0:1].to(device)
        mu, _ = model.encode(x)
        base_z = mu.clone()

    traversal_range = torch.linspace(-3, 3, n_steps)
    fig, axes = plt.subplots(n_dims, n_steps, figsize=(14, n_dims * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for dim in range(n_dims):
        for step_idx, val in enumerate(traversal_range):
            z = base_z.clone()
            z[0, dim] = val
            with torch.no_grad():
                x_hat = model.decoder(z)
            img = x_hat[0].cpu().reshape(28, 28).clamp(0, 1)
            axes[dim, step_idx].imshow(img, cmap="gray")
            axes[dim, step_idx].axis("off")
            if dim == 0:
                axes[dim, step_idx].set_title(f"z={val:.1f}", fontsize=7)
        axes[dim, 0].set_ylabel(f"dim {dim}", fontsize=8, rotation=0, labelpad=30)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_latent_traversal.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_timeseries_reconstruction(model, test_data, title, n_series=4):
    """Overlay original vs reconstructed time series."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n_series].to(device)
        x_hat, _ = model(x)

    fig, axes = plt.subplots(n_series, 1, figsize=(14, 3 * n_series))
    if n_series == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n_series):
        orig = x[i].cpu().numpy()
        recon = x_hat[i].cpu().numpy()
        t = np.arange(len(orig))

        axes[i].plot(t, orig, "b-", linewidth=1.5, label="Original", alpha=0.8)
        axes[i].plot(t, recon, "r--", linewidth=1.5, label="Reconstructed", alpha=0.8)
        axes[i].set_ylabel(f"Series {i + 1}")
        axes[i].legend(loc="upper right", fontsize=8)
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time Step")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_recurrent_ae_timeseries.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


# ════════════════════════════════════════════════════════════════════════
# TRAINING LOOP — shared by all variants
# ════════════════════════════════════════════════════════════════════════

# Collect results across variants (populated by train_variant)
all_losses: dict[str, list[float]] = {}
all_models: dict[str, nn.Module] = {}


async def _train_variant_async(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Universal training loop for any AE variant."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses: list[float] = []

    params = {
        "model_type": name,
        "latent_dim": str(LATENT_DIM),
        "epochs": str(epochs),
        "lr": str(lr),
        "dataset_size": str(len(loader.dataset)),
        "batch_size": str(loader.batch_size),
    }
    if extra_params:
        params.update(extra_params)

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(params)

        for epoch in range(epochs):
            batch_losses = []
            for (xb,) in loader:
                opt.zero_grad()
                loss, _ = loss_fn(model, xb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
            epoch_loss = float(np.mean(batch_losses))
            losses.append(epoch_loss)
            await run.log_metric("loss", epoch_loss, step=epoch + 1)
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  [{name}] epoch {epoch + 1}/{epochs}  loss={epoch_loss:.4f}")
        await run.log_metric("final_loss", losses[-1])

    return losses


def train_variant(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Sync wrapper for training with ExperimentTracker integration."""
    losses = asyncio.run(
        _train_variant_async(
            tracker, exp_name, model, name, loader, loss_fn, epochs, lr, extra_params
        )
    )
    all_losses[name] = losses
    all_models[name] = model
    return losses


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
):
    """Register a single model variant."""
    from kailash_ml.types import MetricSpec

    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=f"m5_{name}",
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="latent_dim", value=float(LATENT_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, loss={final_loss:.4f}")
    return version


def register_model(
    registry: ModelRegistry, name: str, model: nn.Module, final_loss: float
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, final_loss))




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 1.6: Convolutional Autoencoder (Spatial Hierarchy)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Build a Conv AE that preserves spatial locality with Conv2d/ConvTranspose2d
#   - Understand WHY conv layers beat flat MLPs for image data
#   - Observe sharper reconstructions than any flat variant
#   - Apply to e-commerce image compression at Shopee (Conv AE vs JPEG)
#   - Quantify bandwidth cost savings for 50M images/day
#
# PREREQUISITES: 05_contractive_ae.py
# ESTIMATED TIME: ~20 min
#
# TASKS:
#   1. Build Conv AE: 1x28x28 -> 16x14x14 -> 32x7x7 -> latent -> reconstruct
#   2. Train on Fashion-MNIST (image format, not flattened)
#   3. Compare reconstruction sharpness to flat variants
#   4. Apply: image compression rate-distortion vs JPEG
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import io
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image



## THEORY — Spatial Hierarchy via Convolution


In [ ]:
# Conv layers preserve SPATIAL LOCALITY that flat MLPs destroy. A Conv2d
# filter detects patterns (edges, textures) at each spatial position.
# The encoder progressively downsamples: 28x28 -> 14x14 -> 7x7.
#
# Analogy: A flat MLP treats every pixel independently — like reading
# a newspaper by cutting out individual letters and sorting them
# alphabetically. A Conv layer reads the newspaper as-is, detecting
# words, sentences, and paragraphs in their spatial context.
#
# WHY THIS MATTERS: For any image data (product photos, satellite
# imagery, medical scans), spatial relationships carry meaning. A
# button next to a collar means "shirt"; the same button floating
# in space means nothing. Conv AEs preserve these relationships.



## TASK 1 — Load data and engines


In [ ]:
X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader = load_fashion_mnist()
conn, tracker, exp_name, registry, has_registry = setup_engines()



## TASK 2 — Build and Train Convolutional AE


In [ ]:
class ConvAE(nn.Module):
    def __init__(self, latent_dim: int = 16):
        super().__init__()
        # TODO: Build encoder — nn.Sequential:
        #       Conv2d(1, 16, kernel_size=3, stride=2, padding=1), ReLU,
        #       Conv2d(16, 32, kernel_size=3, stride=2, padding=1), ReLU,
        #       Flatten(), Linear(32*7*7, latent_dim)
        self.encoder = ____

        # TODO: Build decoder — nn.Sequential:
        #       Linear(latent_dim, 32*7*7), ReLU,
        #       Unflatten(1, (32, 7, 7)),
        #       ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1), ReLU,
        #       ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1), Sigmoid
        self.decoder = ____

    def forward(self, x):
        # TODO: Encode then decode. Return (reconstruction, latent_code)
        ____


def conv_ae_loss(model, xb):
    # TODO: Forward, MSE loss. Return (loss, {})
    ____


print("\n" + "=" * 70)
print("  Convolutional AE — Spatial Hierarchy")
print("=" * 70)
print("  Conv2d preserves spatial structure. Expect sharper reconstructions.")

# TODO: Create ConvAE(LATENT_DIM) and train on img_loader (not flat_loader!)
conv_model = ____
conv_losses = ____



## TASK 3 — Visualise


In [ ]:
# TODO: show_reconstruction with is_conv=True
____



### Checkpoint


In [ ]:
assert len(conv_losses) == EPOCHS
assert conv_losses[-1] < conv_losses[0]
# INTERPRETATION: Compare to the undercomplete AE. The Conv version
# preserves EDGES and TEXTURES better — sharper outlines of shirts,
# shoes, bags. This is because Conv2d filters share parameters across
# spatial positions, learning translation-invariant features.
print("\n--- Checkpoint passed --- convolutional AE trained\n")

if has_registry:
    register_model(registry, "conv_ae", conv_model, conv_losses[-1])



## APPLY — E-Commerce Image Compression (Shopee)


In [ ]:
# BUSINESS SCENARIO: You are an ML engineer at a Singapore e-commerce
# platform (Shopee/Lazada). The platform serves 50M product images per
# day. Bandwidth costs are S$300K/month. Your VP asks: "Can ML-based
# compression reduce bandwidth costs while maintaining image quality?"

print("\n" + "=" * 70)
print("  APPLICATION: Image Compression vs JPEG (Shopee)")
print("=" * 70)

IMG_SIZE = 28
ORIGINAL_BYTES = IMG_SIZE * IMG_SIZE


def compute_ssim(img1, img2, C1=0.01**2, C2=0.03**2):
    mu1, mu2 = img1.mean(), img2.mean()
    sigma1_sq, sigma2_sq = img1.var(), img2.var()
    sigma12 = ((img1 - mu1) * (img2 - mu2)).mean()
    return float(
        ((2 * mu1 * mu2 + C1) * (2 * sigma12 + C2))
        / ((mu1**2 + mu2**2 + C1) * (sigma1_sq + sigma2_sq + C2))
    )


def compute_psnr(img1, img2):
    mse = np.mean((img1 - img2) ** 2)
    return 10 * np.log10(1.0 / mse) if mse > 0 else float("inf")


# --- JPEG baseline ---
test_images_np = X_test_img[:200].cpu().numpy()[:, 0]
jpeg_qualities = [5, 10, 15, 20, 30, 40, 50, 60, 70, 80, 90, 95]
jpeg_results = []

print("\nJPEG compression baseline...")
for quality in jpeg_qualities:
    ssim_vals, psnr_vals, byte_sizes = [], [], []
    for img in test_images_np[:100]:
        pil_img = Image.fromarray((img * 255).astype(np.uint8), mode="L")
        buf = io.BytesIO()
        pil_img.save(buf, format="JPEG", quality=quality)
        compressed_size = buf.tell()
        buf.seek(0)
        decompressed = np.array(Image.open(buf)).astype(np.float32) / 255.0
        ssim_vals.append(compute_ssim(img, decompressed))
        psnr_vals.append(compute_psnr(img, decompressed))
        byte_sizes.append(compressed_size)
    ratio = ORIGINAL_BYTES / np.mean(byte_sizes)
    jpeg_results.append(
        (ratio, np.mean(ssim_vals), np.mean(psnr_vals), np.mean(byte_sizes), quality)
    )
    print(f"  JPEG q={quality:2d}: ratio={ratio:.1f}x, SSIM={np.mean(ssim_vals):.4f}")


# --- Conv AE at multiple bottleneck sizes ---
class CompressionAE(nn.Module):
    def __init__(self, bottleneck_channels: int):
        super().__init__()
        self.bottleneck_channels = bottleneck_channels
        # TODO: Build encoder — Conv2d(1,16,3,stride=2,padding=1), ReLU,
        #       Conv2d(16,32,3,stride=2,padding=1), ReLU,
        #       Conv2d(32, bottleneck_channels, 3, padding=1), ReLU
        self.encoder = ____

        # TODO: Build decoder — ConvTranspose2d mirroring encoder, end with Sigmoid
        self.decoder = ____

    def forward(self, x):
        # TODO: Return decoder(encoder(x))
        ____

    @property
    def compressed_bytes(self):
        return self.bottleneck_channels * 7 * 7

    @property
    def compression_ratio(self):
        return ORIGINAL_BYTES / self.compressed_bytes


bottleneck_configs = [1, 2, 4, 8, 16]
ae_results = []
ae_models = {}

print("\nTraining Conv AE at different bottleneck sizes...")
for bn_ch in bottleneck_configs:
    # TODO: Create CompressionAE(bn_ch), train 30 epochs on img_loader
    # Evaluate SSIM/PSNR on test set. Store results.
    comp_model = ____
    comp_opt = ____
    for epoch in range(30):
        comp_model.train()
        for (batch,) in img_loader:
            # TODO: Forward, MSE loss, backprop
            ____
    comp_model.eval()
    with torch.no_grad():
        test_recon = comp_model(X_test_img[:200]).cpu().numpy()[:, 0]
    ssim_vals = [compute_ssim(test_images_np[i], test_recon[i]) for i in range(100)]
    psnr_vals = [compute_psnr(test_images_np[i], test_recon[i]) for i in range(100)]
    ae_results.append(
        (
            comp_model.compression_ratio,
            np.mean(ssim_vals),
            np.mean(psnr_vals),
            comp_model.compressed_bytes,
            bn_ch,
        )
    )
    ae_models[bn_ch] = comp_model
    print(
        f"  AE bn={bn_ch:2d}ch: ratio={comp_model.compression_ratio:.1f}x, SSIM={np.mean(ssim_vals):.4f}"
    )

# --- Visualisation 1: Rate-distortion curve ---
# TODO: Plot SSIM and PSNR vs compression ratio for JPEG and AE
# Save to OUTPUT_DIR / "ex1_compression_rate_distortion.png"
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
____
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_compression_rate_distortion.png", dpi=150, bbox_inches="tight"
)
plt.show()

# --- Visualisation 2: Visual comparison grid ---
# TODO: 3-row grid: Original, JPEG at matched ratio, AE at 4ch
# Save to OUTPUT_DIR / "ex1_compression_visual_comparison.png"
fig, axes = plt.subplots(3, 8, figsize=(18, 7))
____
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_compression_visual_comparison.png", dpi=150, bbox_inches="tight"
)
plt.show()

# --- Business Impact ---
DAILY_IMAGES = 50_000_000
MONTHLY_BANDWIDTH_COST = 300_000
savings_pct = 0.15
monthly_savings = MONTHLY_BANDWIDTH_COST * savings_pct
annual_savings = monthly_savings * 12

print("\n" + "=" * 64)
print("BUSINESS IMPACT SUMMARY — E-Commerce Image Compression")
print("=" * 64)
print(f"\nDaily product images served:     {DAILY_IMAGES:>14,}")
print(f"Monthly bandwidth cost:          {'S$' + f'{MONTHLY_BANDWIDTH_COST:,}':>12}")
print(f"\nBandwidth savings/year:          {'S$' + f'{annual_savings:,.0f}':>12}")
print("=" * 64)



## REFLECTION


[x] Built a convolutional AE with stride-2 downsampling/upsampling
  [x] Observed sharper reconstructions than flat MLPs (spatial locality)
  [x] Applied to image compression: Conv AE vs JPEG rate-distortion
  [x] Compared artifact types: AE blur vs JPEG blockiness
  [x] Quantified bandwidth savings for 50M images/day platform

  KEY INSIGHT: Conv2d filters share parameters across spatial positions,
  learning translation-invariant features. A button pattern detected
  at position (5,5) is also detected at (20,20). This parameter sharing
  makes conv AEs dramatically more efficient for image data than flat
  MLPs, which must learn separate weights for each spatial position.

  Next: 07_stacked_ae.py adds depth for hierarchical features...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — five instruments (convolutional stack)


In [ ]:
# First CONV model in the course. The Blood Test now reports grad
# RMS per Conv2d kernel; the X-ray monitors per-channel dead
# fractions. Healthy Conv nets typically have far FEWER dead
# channels than dense nets at equal depth thanks to weight sharing.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _diag_loss(m, batch):
    xb = batch[0] if isinstance(batch, (tuple, list)) else batch
    loss, _ = conv_ae_loss(m, xb)
    return loss


print("\n── Diagnostic Report (Convolutional AE) ──")
diag, findings = run_diagnostic_checkpoint(
    conv_model,
    img_loader,
    _diag_loss,
    title="Convolutional AE",
    n_batches=8,
    train_losses=conv_losses,
    show=False,
)

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
#   [✓] Gradient flow (HEALTHY): min RMS = 8.7e-04 at
#       'encoder.4.weight' (Conv2d). Convolutional weight sharing
#       keeps gradients uniform — spread < 10x across 6 Conv layers.
#   [!] Dead neurons  (WARNING): 'encoder.1' (relu): 14% dead
#       channels. Each dead channel is an unused FILTER — worse
#       than a dead Linear neuron because it wastes spatial capacity.
#   [✓] Loss trend    (HEALTHY): train slope -3.4e-03/epoch.
#       Final loss ~0.0048 — lower than dense AEs at matched
#       latent size because spatial priors make the task easier.



## Final train loss: ~0.0048 after 10 epochs, bottleneck=64 channels.

STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:

 [BLOOD TEST — CONV-SPECIFIC] Gradient spread <10x across all
    Conv layers is the CNN health signature. Slide 5J shows
    why: weight sharing means each filter receives gradient
    from every spatial position, so vanishing is intrinsically
    harder than in dense nets. Contrast 07_stacked where a
    5-layer DENSE net routinely spans 1000x in RMS.
    >> Prescription: If you see a >100x spread, check the
       pooling/stride layout. A stride-2 Conv followed by a
       stride-2 Conv halves spatial dims twice, starving deep
       filters of gradient contributors. Replace with one
       stride-2 + one stride-1.

 [X-RAY — CONV-SPECIFIC] 14% dead channels means 14% of
    filters are permanently off. Each dead filter is an
    unused 3x3 kernel (9 parameters + activations) — wasted
    both in FLOPs and in representation. Worse than dead
    Linear neurons because a CNN's whole premise is that
    each filter specialises in one feature.
    >> Prescription: GELU or LeakyReLU for the encoder stack.
       Or: reduce bottleneck_channels if capacity is excess
       (fewer filters, fewer dead ones). You'll see this
       fix applied in ex_2's ResNet-SE (variant 02).

 [STETHOSCOPE] Final loss ~0.0048 is LOWER than 02 undercomplete
    (~0.025) and LOWER than 07 stacked (~0.018). Why? The 2D
    convolutional prior (translation invariance, local
    connectivity) matches the spatial structure of Fashion-MNIST.
    Lesson: architecture encodes assumptions — Conv says "pixels
    near each other are correlated".
    >> Prescription: No fix. This is the reward for matching
       inductive bias to data geometry.

 FIVE-INSTRUMENT TAKEAWAY: Conv-AEs show what "healthy deep
 network" looks like when the architecture matches the data —
 uniform gradients, low dead%, low loss. You will use this
 reference when comparing to the pathological patterns in
 ex_3 (RNN gradient collapse) and ex_6 (GNN over-smoothing).


## TASK 3 — Visualise


In [ ]:
show_reconstruction(conv_model, X_test_img, "Convolutional AE", is_conv=True)



### Checkpoint


In [ ]:
assert len(conv_losses) == EPOCHS
assert conv_losses[-1] < conv_losses[0]
# INTERPRETATION: Compare to the undercomplete AE. The Conv version
# preserves EDGES and TEXTURES better — sharper outlines of shirts,
# shoes, bags. This is because Conv2d filters share parameters across
# spatial positions, learning translation-invariant features.
print("\n--- Checkpoint passed --- convolutional AE trained\n")

if has_registry:
    register_model(registry, "conv_ae", conv_model, conv_losses[-1])



## APPLY — E-Commerce Image Compression (Shopee)


In [ ]:
# BUSINESS SCENARIO: You are an ML engineer at a Singapore e-commerce
# platform (Shopee/Lazada). The platform serves 50M product images per
# day. Bandwidth costs are S$300K/month. Your VP asks: "Can ML-based
# compression reduce bandwidth costs while maintaining image quality?"

print("\n" + "=" * 70)
print("  APPLICATION: Image Compression vs JPEG (Shopee)")
print("=" * 70)

IMG_SIZE = 28
ORIGINAL_BYTES = IMG_SIZE * IMG_SIZE


def compute_ssim(img1, img2, C1=0.01**2, C2=0.03**2):
    mu1, mu2 = img1.mean(), img2.mean()
    sigma1_sq, sigma2_sq = img1.var(), img2.var()
    sigma12 = ((img1 - mu1) * (img2 - mu2)).mean()
    return float(
        ((2 * mu1 * mu2 + C1) * (2 * sigma12 + C2))
        / ((mu1**2 + mu2**2 + C1) * (sigma1_sq + sigma2_sq + C2))
    )


def compute_psnr(img1, img2):
    mse = np.mean((img1 - img2) ** 2)
    return 10 * np.log10(1.0 / mse) if mse > 0 else float("inf")


# --- JPEG baseline ---
test_images_np = X_test_img[:200].cpu().numpy()[:, 0]
jpeg_qualities = [5, 10, 15, 20, 30, 40, 50, 60, 70, 80, 90, 95]
jpeg_results = []

print("\nJPEG compression baseline...")
for quality in jpeg_qualities:
    ssim_vals, psnr_vals, byte_sizes = [], [], []
    for img in test_images_np[:100]:
        pil_img = Image.fromarray((img * 255).astype(np.uint8), mode="L")
        buf = io.BytesIO()
        pil_img.save(buf, format="JPEG", quality=quality)
        compressed_size = buf.tell()
        buf.seek(0)
        decompressed = np.array(Image.open(buf)).astype(np.float32) / 255.0
        ssim_vals.append(compute_ssim(img, decompressed))
        psnr_vals.append(compute_psnr(img, decompressed))
        byte_sizes.append(compressed_size)
    ratio = ORIGINAL_BYTES / np.mean(byte_sizes)
    jpeg_results.append(
        (ratio, np.mean(ssim_vals), np.mean(psnr_vals), np.mean(byte_sizes), quality)
    )
    print(f"  JPEG q={quality:2d}: ratio={ratio:.1f}x, SSIM={np.mean(ssim_vals):.4f}")


# --- Conv AE at multiple bottleneck sizes ---
class CompressionAE(nn.Module):
    def __init__(self, bottleneck_channels: int):
        super().__init__()
        self.bottleneck_channels = bottleneck_channels
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, bottleneck_channels, 3, padding=1),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(bottleneck_channels, 32, 3, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

    @property
    def compressed_bytes(self):
        return self.bottleneck_channels * 7 * 7

    @property
    def compression_ratio(self):
        return ORIGINAL_BYTES / self.compressed_bytes


bottleneck_configs = [1, 2, 4, 8, 16]
ae_results = []
ae_models = {}

print("\nTraining Conv AE at different bottleneck sizes...")
for bn_ch in bottleneck_configs:
    comp_model = CompressionAE(bn_ch).to(device)
    comp_opt = torch.optim.Adam(comp_model.parameters(), lr=1e-3)
    for epoch in range(30):
        comp_model.train()
        for (batch,) in img_loader:
            recon = comp_model(batch)
            loss = F.mse_loss(recon, batch)
            comp_opt.zero_grad()
            loss.backward()
            comp_opt.step()
    comp_model.eval()
    with torch.no_grad():
        test_recon = comp_model(X_test_img[:200]).cpu().numpy()[:, 0]
    ssim_vals = [compute_ssim(test_images_np[i], test_recon[i]) for i in range(100)]
    psnr_vals = [compute_psnr(test_images_np[i], test_recon[i]) for i in range(100)]
    ae_results.append(
        (
            comp_model.compression_ratio,
            np.mean(ssim_vals),
            np.mean(psnr_vals),
            comp_model.compressed_bytes,
            bn_ch,
        )
    )
    ae_models[bn_ch] = comp_model
    print(
        f"  AE bn={bn_ch:2d}ch: ratio={comp_model.compression_ratio:.1f}x, SSIM={np.mean(ssim_vals):.4f}"
    )

# --- Visualisation 1: Rate-distortion curve ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
jpeg_ratios = [r[0] for r in jpeg_results]
jpeg_ssims = [r[1] for r in jpeg_results]
ae_ratios = [r[0] for r in ae_results]
ae_ssims = [r[1] for r in ae_results]

axes[0].plot(
    jpeg_ratios,
    jpeg_ssims,
    "o-",
    color="#F44336",
    linewidth=2,
    markersize=6,
    label="JPEG",
)
axes[0].plot(
    ae_ratios,
    ae_ssims,
    "s-",
    color="#2196F3",
    linewidth=2,
    markersize=6,
    label="Conv AE",
)
axes[0].set_xlabel("Compression Ratio (x)")
axes[0].set_ylabel("SSIM (higher = better)")
axes[0].set_title("Rate-Distortion: Conv AE vs JPEG", fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0.5, 1.02)

jpeg_psnrs = [r[2] for r in jpeg_results]
ae_psnrs = [r[2] for r in ae_results]
axes[1].plot(
    jpeg_ratios,
    jpeg_psnrs,
    "o-",
    color="#F44336",
    linewidth=2,
    markersize=6,
    label="JPEG",
)
axes[1].plot(
    ae_ratios,
    ae_psnrs,
    "s-",
    color="#2196F3",
    linewidth=2,
    markersize=6,
    label="Conv AE",
)
axes[1].set_xlabel("Compression Ratio (x)")
axes[1].set_ylabel("PSNR (dB)")
axes[1].set_title("Rate-Distortion: PSNR", fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_compression_rate_distortion.png", dpi=150, bbox_inches="tight"
)
plt.show()

# --- Visualisation 2: Visual comparison grid ---
ae_compare = ae_models[4]
ae_compare.eval()
target_ratio = ae_compare.compression_ratio
jpeg_idx = np.argmin([abs(r[0] - target_ratio) for r in jpeg_results])
jpeg_q = jpeg_results[jpeg_idx][4]

fig, axes = plt.subplots(3, 8, figsize=(18, 7))
with torch.no_grad():
    ae_recon_np = ae_compare(X_test_img[:8]).cpu().numpy()[:, 0]

for i in range(8):
    orig = test_images_np[i]
    pil_img = Image.fromarray((orig * 255).astype(np.uint8), mode="L")
    buf = io.BytesIO()
    pil_img.save(buf, format="JPEG", quality=jpeg_q)
    buf.seek(0)
    jpeg_img = np.array(Image.open(buf)).astype(np.float32) / 255.0

    axes[0, i].imshow(orig, cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title("Original", fontsize=9)
    axes[0, i].axis("off")
    axes[1, i].imshow(jpeg_img, cmap="gray", vmin=0, vmax=1)
    axes[1, i].set_title(
        f"JPEG q={jpeg_q}\nSSIM={compute_ssim(orig, jpeg_img):.3f}", fontsize=8
    )
    axes[1, i].axis("off")
    axes[2, i].imshow(ae_recon_np[i], cmap="gray", vmin=0, vmax=1)
    axes[2, i].set_title(
        f"AE 4ch\nSSIM={compute_ssim(orig, ae_recon_np[i]):.3f}", fontsize=8
    )
    axes[2, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=11, rotation=0, labelpad=50)
axes[1, 0].set_ylabel("JPEG", fontsize=11, rotation=0, labelpad=50)
axes[2, 0].set_ylabel("Conv AE", fontsize=11, rotation=0, labelpad=50)
fig.suptitle(f"Visual Comparison at ~{target_ratio:.0f}x Compression", fontsize=13)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_compression_visual_comparison.png", dpi=150, bbox_inches="tight"
)
plt.show()

# --- Business Impact ---
DAILY_IMAGES = 50_000_000
MONTHLY_BANDWIDTH_COST = 300_000
ae_4ch_ssim = [r[1] for r in ae_results if r[4] == 4][0]
jpeg_matched_ssim = jpeg_results[jpeg_idx][1]
savings_pct = 0.15
monthly_savings = MONTHLY_BANDWIDTH_COST * savings_pct
annual_savings = monthly_savings * 12

print("\n" + "=" * 64)
print("BUSINESS IMPACT SUMMARY — E-Commerce Image Compression")
print("=" * 64)
print(f"\nDaily product images served:     {DAILY_IMAGES:>14,}")
print(f"Monthly bandwidth cost:          {'S$' + f'{MONTHLY_BANDWIDTH_COST:,}':>12}")
print(f"\nAt ~{target_ratio:.0f}x compression:")
print(f"  JPEG SSIM:  {jpeg_matched_ssim:.4f}")
print(f"  AE SSIM:    {ae_4ch_ssim:.4f}  (+{ae_4ch_ssim - jpeg_matched_ssim:.4f})")
print(f"\nBandwidth savings/year:          {'S$' + f'{annual_savings:,.0f}':>12}")
print(f"  AE: smoother blur artifacts (preserves edges)")
print(f"  JPEG: blocky 8x8 grid artifacts")
print("=" * 64)



## REFLECTION


[x] Built a convolutional AE with stride-2 downsampling/upsampling
  [x] Observed sharper reconstructions than flat MLPs (spatial locality)
  [x] Applied to image compression: Conv AE vs JPEG rate-distortion
  [x] Compared artifact types: AE blur vs JPEG blockiness
  [x] Quantified bandwidth savings for 50M images/day platform

  KEY INSIGHT: Conv2d filters share parameters across spatial positions,
  learning translation-invariant features. A button pattern detected
  at position (5,5) is also detected at (20,20). This parameter sharing
  makes conv AEs dramatically more efficient for image data than flat
  MLPs, which must learn separate weights for each spatial position.

  Next: 07_stacked_ae.py adds depth for hierarchical features...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

